# 캐글 대회 - 예지보전
### FType이 답지 → 오입력 패턴 탐지 + 보정

In [8]:
# 모듈 임포트
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from scipy.stats import rankdata

In [9]:
# 데이터 로드
os.chdir('../data')
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

## 전처리

### 오입력 케이스 분류
| 케이스 | Target | Failure Type | 의미 | 건수 | 처리 |
|--------|--------|-------------|------|------|------|
| A | 0 | No Failure | 정상 → 정상 ✓ | 9643건 | 정상 데이터 |
| B | 1 | No Failure | 고장인데 FType 누락 | 9건 | 수정 안 함 (실제 고장) |
| C | 0 | Failure (RF) | 정상인데 FType 오입력 | 18건 | train에서 제거 (노이즈) |
| D | 1 | Failure | 고장 → 고장 ✓ | 330건 | 정상 데이터 |

- Case C는 전부 Random Failures (다른 FType 오입력 0건)
- test RF 7건 → 예측값 0.0으로 후처리 보정

In [10]:
# Case C 노이즈 제거
train = train[train['Failure Type'] != 'Random Failures'].copy()

In [11]:
# 타겟 분리
yvar = 'Target'
y = train[yvar].copy()

# 피처 선택
feature_cols = ['Type', 'Air temperature [K]', 'Process temperature [K]',
                'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Failure Type']
X = train[feature_cols].copy()
X_test = test[feature_cols].copy()

# 컬럼명 단축
col_names = ['Type', 'AirTmp', 'ProcTmp', 'RotSpd', 'Torque', 'ToolWear', 'FType']
X.columns = col_names
X_test.columns = col_names

# Type 인코딩
oe_type = OrdinalEncoder(categories=[['L', 'M', 'H']])
X['Type'] = oe_type.fit_transform(X[['Type']])
X_test['Type'] = oe_type.transform(X_test[['Type']])

# Failure Type 인코딩
ft_cats = ['No Failure', 'Heat Dissipation Failure', 'Overstrain Failure',
           'Power Failure', 'Random Failures', 'Tool Wear Failure']
oe_ft = OrdinalEncoder(categories=[ft_cats])
X['FType'] = oe_ft.fit_transform(X[['FType']])
X_test['FType'] = oe_ft.transform(X_test[['FType']])

In [12]:
# 파생변수 추가
def add_features(df):
    df = df.copy()
    df['temp_diff'] = df['ProcTmp'] - df['AirTmp']
    df['power'] = df['Torque'] * df['RotSpd']
    df['wear_torque'] = df['ToolWear'] * df['Torque']
    df['wear_speed'] = df['ToolWear'] * df['RotSpd']
    df['temp_per_torque'] = df['temp_diff'] / (df['Torque'] + 1e-6)
    df['temp_per_power'] = df['temp_diff'] / (df['power'] + 1e-6)
    return df

In [13]:
X = add_features(X)
X_test = add_features(X_test)

## 5모델 앙상블

In [14]:
n_seeds = 50
n_splits = 5

def get_models(seed):
    pos_weight = len(y[y==0]) / len(y[y==1])
    return {
        'lgbm': LGBMClassifier(
            class_weight='balanced', random_state=seed, n_jobs=-1, verbose=-1,
            num_leaves=15, max_depth=5, min_child_samples=30,
            reg_alpha=0.1, reg_lambda=1.0, subsample=0.8, colsample_bytree=0.8,
            n_estimators=200,
        ),
        'xgb': XGBClassifier(
            scale_pos_weight=pos_weight, random_state=seed, n_jobs=-1,
            max_depth=4, min_child_weight=30, reg_alpha=0.1, reg_lambda=1.0,
            subsample=0.8, colsample_bytree=0.8, n_estimators=200,
            verbosity=0,
        ),
        'rf': RandomForestClassifier(
            class_weight='balanced', random_state=seed, n_jobs=-1,
            max_depth=8, min_samples_leaf=20, n_estimators=200,
        ),
        'cat': CatBoostClassifier(
            auto_class_weights='Balanced', random_seed=seed,
            depth=5, l2_leaf_reg=3.0, subsample=0.8,
            iterations=200, verbose=0,
        ),
        'et': ExtraTreesClassifier(
            class_weight='balanced', random_state=seed, n_jobs=-1,
            max_depth=8, min_samples_leaf=20, n_estimators=200,
        ),
    }

model_names = ['lgbm', 'xgb', 'rf', 'cat', 'et']
model_preds = {name: [] for name in model_names}

for seed in range(n_seeds):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    models = get_models(seed)
    for name, m in models.items():
        fold_preds = np.zeros(len(X_test))
        for train_idx, _ in skf.split(X, y):
            m.fit(X.iloc[train_idx], y.iloc[train_idx])
            fold_preds += m.predict_proba(X_test)[:, 1] / n_splits
        model_preds[name].append(rankdata(fold_preds) / len(fold_preds))
    
    if (seed + 1) % 10 == 0:
        print(f'  {seed + 1}/{n_seeds} seeds done...')

for name in model_preds:
    model_preds[name] = np.mean(model_preds[name], axis=0)
    print(f'{name} 예측 범위: {model_preds[name].min():.4f} ~ {model_preds[name].max():.4f}')

ensemble_pred = np.mean(list(model_preds.values()), axis=0)
print(f'\n앙상블 완료: {len(model_names)}모델 × {n_seeds}시드 × {n_splits}폴드 = {len(model_names)*n_seeds*n_splits}개 모델')

  10/50 seeds done...
  20/50 seeds done...
  30/50 seeds done...
  40/50 seeds done...
  50/50 seeds done...
lgbm 예측 범위: 0.0023 ~ 0.9996
xgb 예측 범위: 0.0011 ~ 1.0000
rf 예측 범위: 0.0105 ~ 1.0000
cat 예측 범위: 0.0016 ~ 0.9997
et 예측 범위: 0.0035 ~ 0.9997

앙상블 완료: 5모델 × 50시드 × 5폴드 = 1250개 모델


## RF 보정 및 저장

In [15]:
# 제출: 5모델 앙상블 + RF 보정
submit = pd.read_csv('sample_submission.csv')
rf_mask = (X_test['FType'] == 4).values

# RF 7건 → 0 보정 (train RF 100% Target=0 근거)
final_pred = ensemble_pred.copy()
final_pred[rf_mask] = 0.0
submit[yvar] = final_pred
submit.to_csv('v11b_rf0.csv', index=False)

print(f'v11b: 5모델 50시드 앙상블 + RF {rf_mask.sum()}건 보정')

v11b: 5모델 50시드 앙상블 + RF 7건 보정


## 성능 지표 확인

In [ ]:
# # OOF 예측값 수집
# from sklearn.metrics import f1_score, classification_report, confusion_matrix
# from sklearn.model_selection import StratifiedKFold

# oof_preds_sum = np.zeros(len(X))
# oof_counts = np.zeros(len(X))

# for seed in range(n_seeds):
#     skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
#     models = get_models(seed)
#     for name, m in models.items():
#         for train_idx, val_idx in skf.split(X, y):
#             m.fit(X.iloc[train_idx], y.iloc[train_idx])
#             oof_preds_sum[val_idx] += m.predict_proba(X.iloc[val_idx])[:, 1]
#             oof_counts[val_idx] += 1

# oof_avg = oof_preds_sum / oof_counts

# # 최적 threshold 탐색
# best_f1, best_thr = 0, 0.5
# for thr in np.arange(0.1, 0.9, 0.01):
#     f1 = f1_score(y, (oof_avg >= thr).astype(int))
#     if f1 > best_f1:
#         best_f1, best_thr = f1, thr

# y_pred = (oof_avg >= best_thr).astype(int)

# print(f'최적 Threshold: {best_thr:.2f}')
# print(f'OOF F1 Score: {best_f1:.4f}')
# print()
# print(classification_report(y, y_pred, target_names=['정상(0)', '고장(1)']))
# print('Confusion Matrix:')
# print(confusion_matrix(y, y_pred))

최적 Threshold: 0.35
OOF F1 Score: 0.9865

              precision    recall  f1-score   support

       정상(0)       1.00      1.00      1.00      9643
       고장(1)       1.00      0.97      0.99       339

    accuracy                           1.00      9982
   macro avg       1.00      0.99      0.99      9982
weighted avg       1.00      1.00      1.00      9982

Confusion Matrix:
[[9643    0]
 [   9  330]]
